# imdb sentiment analysis
**author:** maxime  
**reviewer:** hugo

## 1. business & data understanding
my goal here is pretty simple: train a model to guess if a movie review is positive or negative. i picked naive bayes (multinomialnb specifically) because honestly it's one of the best out-of-the-box algorithms when you're dealing with text and nlp stuff. 

quick side note: the original dataset with 50k reviews was way too heavy to host on github (25mb limit). so i just took a random sample of 10,000 rows for this notebook. the full original dataset comes from kaggle and is available here: [imdb dataset of 50k movie reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews).

In [10]:
import pandas as pd

# load data
df = pd.read_csv('IMDB_Dataset_mini.csv')

# show first rows
df.head()

,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


## 2. data preparation
since machine learning models can't actually read words, i had to change the text into numbers using countvectorizer. basically it just counts how many times each word appears. i also mapped the sentiment tags to 1 for positive and 0 for negative.

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# tags: 'positive' -> 1, 'negative' -> 0
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# set X and y
X = df['review']
y = df['sentiment']

# 80/20 train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# vectorize the text
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

## 3. modeling & evaluation
alright so now i'm fitting multinomialnb to the vectorized text. after that i'm running a quick accuracy test and then trying it out on my own custom review at the end just to see if it really works.

In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# train multinomial naive bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vectorized, y_train)

# predict on test set
y_pred = nb_model.predict(X_test_vectorized)

# print scores
print("accuracy score:", accuracy_score(y_test, y_pred))
print("\nclassification report:\n", classification_report(y_test, y_pred))

# custom test
test_review = ["this movie was absolutely fantastic! i loved it."]
test_review_vectorized = vectorizer.transform(test_review)
prediction = nb_model.predict(test_review_vectorized)

print("\n--- custom prediction ---")
print("review:", test_review[0])
print("prediction:", 'positive' if prediction[0] == 1 else 'negative')

accuracy score: 0.829

classification report:
               precision    recall  f1-score   support

           0       0.80      0.88      0.84       999
           1       0.86      0.78      0.82      1001

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.83      2000


--- custom prediction ---
review: this movie was absolutely fantastic! i loved it.
prediction: positive


## 4. peer review (by hugo)
yo maxime, the notebook is super clean. i actually love the idea of adding a custom prediction at the end, it really proves the model works in real life haha. as an improvement for next time, you could maybe use a tfidfvectorizer instead of countvectorizer, so that super common words don't get too much weight and ruin the predictions.